# Task 5: real versus synthetic noise

The `synthetic_label` column has symmetric noise: about the same overall rate as the human labels, but each wrong label is a uniformly random other class. Two models are retrained on it with exactly the Task 1 / Task 4 setup:

* the cross-entropy baseline (60 epochs, final-epoch predictions, as in Task 1), with the same per-epoch logging as Task 2;
* the best Task 4 method, run through the same pipeline as on the human labels. If that method needs the out-of-fold predictions (relabel, or the noise-rate estimate for co-teaching), the 5-fold cross-validation of Task 3 is repeated on the synthetic labels first.

Model selection uses the synthetic validation labels only. Clean labels are used only to compare the two noise types on equal terms (clean validation accuracy, fit to wrong labels, detection AUROC).

The other two Task 4 methods are also run on the synthetic labels, only to compare the ranking of methods (their predictions are not submitted).

Runtime on an A100: about 45 minutes.

In [1]:
# project folder: MyDrive/cv_bootcamp (data/, model.py, check_format.py, results_template.json, code/cvlib.py).
# If it is not there yet, it is unpacked from MyDrive/cv_bootcamp.zip, or from a zip picked in the file dialog.
import sys, zipfile
from pathlib import Path

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/cv_bootcamp")
except ImportError:                      # running outside Colab
    files = None
    ROOT = Path.cwd() / "cv_bootcamp"

if not (ROOT / "data" / "labels.csv").exists():
    zips = [z for z in [ROOT.parent / "cv_bootcamp.zip"] if z.exists()] + list(Path.cwd().glob("cv_bootcamp*.zip"))
    if not zips:
        zips = [Path(name) for name in files.upload()]
    zipfile.ZipFile(zips[0]).extractall(ROOT.parent)
    print("unpacked", zips[0].name, "into", ROOT.parent)

assert (ROOT / "data" / "labels.csv").exists() and (ROOT / "code" / "cvlib.py").exists()
sys.path.insert(0, str(ROOT / "code"))
print("project folder:", ROOT)

Mounted at /content/drive
project folder: /content/drive/MyDrive/cv_bootcamp


In [2]:
import json
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import cvlib as cv

WORK = ROOT / "work"
PRED = ROOT / "predictions"
WORK.mkdir(exist_ok=True)
PRED.mkdir(exist_ok=True)

print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
data = cv.Data(ROOT)
print("train", len(data.train_idx), " val", len(data.val_idx), " test", len(data.x_test))
print("classes:", data.classes)

device: NVIDIA A100-SXM4-80GB
train 45000  val 5000  test 10000
classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

lab = data.labels
tr, va = data.train_idx, data.val_idx
noisy, clean, synth = (lab[c].to_numpy() for c in ["noisy_label", "clean_label", "synthetic_label"])
y_syn = synth[tr]
wrong_syn = y_syn != clean[tr]
res = cv.load_results(WORK)
best_method = res["task4"]["best_method"]
print(f"synthetic noise rate (train): {wrong_syn.mean():.4f}   best Task 4 method: {best_method}")

EXTRA_METHODS = True       # also runs the other Task 4 methods on synthetic labels, for the Q5 ranking only (not submitted)

synthetic noise rate (train): 0.4010   best Task 4 method: coteaching


## Cross-entropy baseline on synthetic labels

In [4]:
RUN = WORK / "t5_baseline_synth.npz"
if RUN.exists():
    base = dict(np.load(RUN))
    print("loaded", RUN.name)
else:
    rec = cv.Recorder(data, synth[va])
    train_probs, rows = [], []

    def on_epoch(ep, net):
        info = rec(ep, net)
        p = cv.predict(net, data.x[tr])
        train_probs.append(p.astype(np.float16))
        pred, pv = p.argmax(1), rec.val_probs[-1].argmax(1)
        rows.append({"epoch": ep,
                     "train_acc_correct": float((pred[~wrong_syn] == y_syn[~wrong_syn]).mean()),
                     "train_acc_mislabeled": float((pred[wrong_syn] == y_syn[wrong_syn]).mean()),
                     "val_acc_synth": float((pv == synth[va]).mean()),
                     "val_acc_clean": float((pv == clean[va]).mean())})
        return info

    cv.train(data, tr, y_syn, on_epoch=on_epoch, tag="CE synthetic")
    rec.save(RUN, train_probs=np.stack(train_probs), dynamics=pd.DataFrame(rows).to_numpy(),
             dynamics_cols=np.array(list(rows[0].keys())))
    base = dict(np.load(RUN))

dyn_s = pd.DataFrame(base["dynamics"], columns=[str(c) for c in base["dynamics_cols"]])
cv.write_preds(PRED / "t5_baseline_synth.csv", base["test_probs"][-1].astype(np.float32))
print(f"final epoch: val acc synthetic {dyn_s.val_acc_synth.iloc[-1]:.4f}, clean {dyn_s.val_acc_clean.iloc[-1]:.4f}")

CE synthetic epoch  1  loss 2.3284  val_acc 0.2204  (19s)
CE synthetic epoch  5  loss 2.0014  val_acc 0.3310  (37s)
CE synthetic epoch 10  loss 1.8750  val_acc 0.3756  (59s)
CE synthetic epoch 15  loss 1.8004  val_acc 0.4552  (82s)
CE synthetic epoch 20  loss 1.7606  val_acc 0.4466  (105s)
CE synthetic epoch 25  loss 1.7233  val_acc 0.4640  (128s)
CE synthetic epoch 30  loss 1.6935  val_acc 0.4768  (150s)
CE synthetic epoch 35  loss 1.6504  val_acc 0.4716  (173s)
CE synthetic epoch 40  loss 1.5955  val_acc 0.4946  (196s)
CE synthetic epoch 45  loss 1.5111  val_acc 0.4996  (218s)
CE synthetic epoch 50  loss 1.3500  val_acc 0.4818  (241s)
CE synthetic epoch 55  loss 1.0474  val_acc 0.4514  (263s)
CE synthetic epoch 60  loss 0.8068  val_acc 0.4300  (286s)
final epoch: val acc synthetic 0.4300, clean 0.7116


## Best Task 4 method on synthetic labels

In [ ]:
def run_fold(k, fit, hold, labels, prefix):
    path = WORK / f"{prefix}_fold{k}.npz"
    if path.exists():
        return dict(np.load(path))
    held = []
    rec = cv.Recorder(data, labels[va])

    def on_epoch(ep, net):
        held.append(cv.predict(net, data.x[tr[hold]]).astype(np.float16))
        return rec(ep, net)

    cv.train(data, tr[fit], labels[tr][fit], seed=cv.SEED + k, on_epoch=on_epoch, tag=f"{prefix} fold {k}")
    np.savez_compressed(path, hold=hold, probs=np.stack(held), val_acc=np.array(rec.val_acc))
    return dict(np.load(path))

def oof_for(labels, prefix, K=5, avg_from=10):
    y = labels[tr]
    folds = StratifiedKFold(n_splits=K, shuffle=True, random_state=cv.SEED).split(tr, y)
    oof = np.zeros((len(tr), 10), np.float32)
    for k, (fit, hold) in enumerate(folds):
        f = run_fold(k, fit, hold, labels, prefix)
        best = int(f["val_acc"].argmax()) + 1
        oof[f["hold"]] = f["probs"][min(avg_from, best) - 1:best].astype(np.float32).mean(0)
    return oof

def run_method(name, labels, prefix):
    """The Task 4 pipeline for one method on one label column; selection on that column's val labels."""
    path = WORK / f"{prefix}_{name}.npz"
    if path.exists():
        return dict(np.load(path))
    y = labels[tr]
    rec = cv.Recorder(data, labels[va])
    if name == "sce":
        cv.train(data, tr, y, loss_fn=cv.sce_loss, on_epoch=rec, tag=f"{name} ({prefix})")
    else:
        oof = oof_for(labels, f"{prefix}cv")
        pred, conf = oof.argmax(1), oof.max(1)
        if name == "relabel":
            agree = pred == y
            relab = ~agree & (conf >= 0.5)
            keep = agree | relab
            new = np.where(relab, pred, y)
            print(f"kept {agree.sum()}, relabelled {relab.sum()}, dropped {(~keep).sum()}")
            cv.train(data, tr[keep], new[keep], on_epoch=rec, tag=f"{name} ({prefix})")
        elif name == "coteaching":
            est = float((pred != y).mean())
            print(f"estimated noise rate: {est:.4f}")
            cv.train_coteaching(data, tr, y, forget_rate=est, on_epoch=rec, tag=f"{name} ({prefix})")
    rec.save(path)
    return dict(np.load(path))

best_syn = run_method(best_method, synth, "t5")
acc_s = np.array(best_syn["val_acc"])
best_ep = int(acc_s.argmax()) + 1
cv.write_preds(PRED / "t5_best_synth.csv", best_syn["test_probs"][best_ep - 1].astype(np.float32))
acc_c_best = float((best_syn["val_probs"][best_ep - 1].astype(np.float32).argmax(1) == clean[va]).mean())
print(f"{best_method} on synthetic labels: best epoch {best_ep}, val acc synthetic {acc_s[best_ep - 1]:.4f}, clean {acc_c_best:.4f}")

cv.update_results(WORK, task5={"val_acc_synthetic_labels__baseline": round(float(dyn_s.val_acc_synth.iloc[-1]), 4),
                               "val_acc_synthetic_labels__best_method": round(float(acc_s[best_ep - 1]), 4)})

t5cv fold 0 epoch  1  loss 2.5048  val_acc 0.1718  (8s)
t5cv fold 0 epoch  5  loss 2.1034  val_acc 0.2766  (22s)
t5cv fold 0 epoch 10  loss 1.9912  val_acc 0.3322  (40s)
t5cv fold 0 epoch 15  loss 1.8962  val_acc 0.4008  (57s)
t5cv fold 0 epoch 20  loss 1.8332  val_acc 0.4148  (74s)
t5cv fold 0 epoch 25  loss 1.7740  val_acc 0.4140  (92s)
t5cv fold 0 epoch 30  loss 1.7283  val_acc 0.4616  (109s)
t5cv fold 0 epoch 35  loss 1.6739  val_acc 0.4758  (126s)
t5cv fold 0 epoch 40  loss 1.6143  val_acc 0.4858  (143s)
t5cv fold 0 epoch 45  loss 1.5194  val_acc 0.4760  (160s)
t5cv fold 0 epoch 50  loss 1.3412  val_acc 0.4636  (177s)
t5cv fold 0 epoch 55  loss 1.0073  val_acc 0.4394  (195s)
t5cv fold 0 epoch 60  loss 0.7510  val_acc 0.4204  (213s)
t5cv fold 1 epoch  1  loss 2.4657  val_acc 0.1548  (3s)
t5cv fold 1 epoch  5  loss 2.1021  val_acc 0.2686  (18s)
t5cv fold 1 epoch 10  loss 1.9609  val_acc 0.3230  (36s)
t5cv fold 1 epoch 15  loss 1.8758  val_acc 0.3360  (54s)
t5cv fold 1 epoch 20  loss

{'task1': {'noise_rate': 0.4024,
  'val_acc_noisy_labels__clean_ref_model': 0.5918,
  'val_acc_noisy_labels__noisy_ce_model': 0.4924},
 'seed': 42,
 'training_setup': {'optimizer': 'SGD (momentum 0.9, nesterov, weight decay 5e-4)',
  'learning_rate_schedule': 'cosine annealing per step, 0.1 to 0 over 60 epochs, no warmup',
  'batch_size': 128,
  'epochs': 60,
  'augmentation': 'random crop 32x32 with 4-pixel zero padding, random horizontal flip'},
 'extra': {'clean_ref_val_acc_clean': 0.9446,
  'noisy_ce_val_acc_clean_final': 0.6844,
  'noisy_ce_val_acc_clean_early_stop': 0.779,
  'clean_val_peak_epoch': 43,
  't3_auroc': 0.9418,
  't3_precision_at_k': 0.8307,
  't3_check': {'cv_confidence (submitted)': {'AUROC': 0.9418,
    'precision@k': 0.8307},
   'cv_last_epoch': {'AUROC': 0.8774, 'precision@k': 0.7637},
   'early_loss': {'AUROC': 0.9371, 'precision@k': 0.8172},
   'cv_disagreement': {'AUROC': 0.8834, 'precision@k': 0.8222},
   'combined_rank': {'AUROC': 0.9417, 'precision@k': 0.8

## Comparison with the human noise

Accuracies on the two noisy validation sets are not comparable with each other (different wrong labels), so the comparison uses clean validation accuracy, which is only measured here, never used for selection.

In [ ]:
d = np.load(WORK / "t1_noisy_ce.npz")
dyn_h = pd.DataFrame(d["dynamics"], columns=[str(c) for c in d["dynamics_cols"]])

def clean_acc(run, ep):
    return float((run["val_probs"][ep - 1].astype(np.float32).argmax(1) == clean[va]).mean())

def summary(dyn, val_col, runs):
    es = int(dyn[val_col].to_numpy().argmax()) + 1
    out = {"CE final": dyn.val_acc_clean.iloc[-1], "CE early stop": dyn.val_acc_clean.iloc[es - 1]}
    for name, r in runs.items():
        ep = int(np.argmax(r["val_acc"])) + 1
        out[name] = clean_acc(r, ep)
    return out

human_runs = {n: dict(np.load(WORK / f"t4_{n}.npz")) for n in ["relabel", "sce", "coteaching"] if (WORK / f"t4_{n}.npz").exists()}
synth_runs = {best_method: best_syn}
if EXTRA_METHODS:
    for n in human_runs:
        if n != best_method:
            synth_runs[n] = run_method(n, synth, "t5")

comp = pd.DataFrame({"human noise": summary(dyn_h, "val_acc_noisy", human_runs),
                     "synthetic noise": summary(dyn_s, "val_acc_synth", synth_runs)})
comp["rank human"] = comp["human noise"].rank(ascending=False)
comp["rank synthetic"] = comp["synthetic noise"].rank(ascending=False)
print("clean validation accuracy of each model (selection was done on that model's own noisy labels)")
display(comp.round(4))

In [ ]:
# how fast each kind of wrong label gets memorised, and how easy it is to detect
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(dyn_h.epoch, dyn_h.train_acc_mislabeled, label="human noise")
ax[0].plot(dyn_s.epoch, dyn_s.train_acc_mislabeled, label="synthetic noise")
ax[0].set_title("Fit to wrong training labels")
ax[1].plot(dyn_h.epoch, dyn_h.val_acc_clean, label="human noise")
ax[1].plot(dyn_s.epoch, dyn_s.val_acc_clean, label="synthetic noise")
ax[1].set_title("Clean validation accuracy, cross-entropy")
for a in ax:
    a.set_xlabel("epoch"); a.set_ylabel("accuracy"); a.grid(alpha=0.3); a.legend()
fig.tight_layout(); fig.savefig(WORK / "t5_real_vs_synthetic.png", dpi=150); plt.show()

mem = res["task2"]["early_stop_epoch"]          # same window as the Task 3 early-loss signal
pos = np.arange(len(tr))
def early_loss_auroc(run, y, wrong):
    tp = run["train_probs"][:mem].astype(np.float32)
    return roc_auc_score(wrong, -np.log(np.clip(tp[:, pos, y], 1e-7, 1)).mean(0))

det = {"human": early_loss_auroc(d, noisy[tr], noisy[tr] != clean[tr]),
       "synthetic": early_loss_auroc(base, y_syn, wrong_syn)}
print(f"detection AUROC of the early-loss signal (epochs 1-{mem}): human {det['human']:.4f}, synthetic {det['synthetic']:.4f}")
for name, dd in [("human", dyn_h), ("synthetic", dyn_s)]:
    print(f"{name}: fit to wrong labels at epoch {mem}: {dd.train_acc_mislabeled.iloc[mem - 1]:.4f}, final: {dd.train_acc_mislabeled.iloc[-1]:.4f}")

cv.update_results(WORK, extra={"t5_comparison": comp.round(4).reset_index().to_dict("records"),
                               "t5_early_loss_auroc": {k: round(v, 4) for k, v in det.items()}})
print(json.dumps(cv.load_results(WORK)["task5"], indent=2))